The workflow for a data science project will follow these lines:

1. Get and explore the data
2. Build a model 
3. Train the model
4. Save and predict

## 1. Get and Explore the Data
The first step can take quite some time; data quality is often something that needs to be checked, and correlations between data should often be explored and visualized.

This step can be a full project on its own: you clean the data, make sure you can access it properly, and create visualizations and hypothesis to gain insight into the data that can be shown in a dashboard.

The insight in the data is an essential ingredient for deciding on a model.

## 2. Build a model
Based on domain knowledge and a first exploration of the data, a model can be selected.

Sometimes, the relation between features and outcome is very obvious. You might have features that
correlate very high with the outcome variable, and a domain expert confirms that the correlations make sense.

If this is the case, you can often build a simple model. If you expect to have non-linear and complex interactions between the features,
you could use a model that works with non-linear data like a SVM plus kernel, or a random forest.

If you have enough data (as a rule of thumb, a lower threshold of 1000 observations) you can consider a neural network architecture.
If the expected complexity of the data is low, you can use a relative small network.
If you have lots and lots of data with a high complexity, you should consider to increase the complexity of your model too.

How you can build a model, and what suitable models are for different datatypes and situations, will be the subject of the whole course.

## 3. Train the model
Once you created a model, it hasnt learned anything yet. The model must be trained to learn the right connections, a bit like a baby that has to learn about what works and what doesn't.

In this notebook, I will introduce you to PyTorch. Another high level library is Tensorflow, which is used a lot too.
While the interface is comparable, the Tensorflow syntax is a bit more high-level. While this can be an advantage, 
it also has a downside: at the moment you ever need to dive a bit deeper into the architecture itself, it is much harder to
add something new with TensorFlow, compared to PyTorch.

## 4. Save and predict
Finally, you will want to use the trained model to predict new observations.

# Load the data
We will use the fashion MNIST dataset. You will find this dataset a lot in machine learning tutorials. It are small (28x28) images of clothing.

In [ ]:
from pathlib import Path
import mads_datasets
from mads_datasets import DatasetFactoryProvider, DatasetType
import warnings
from tomlserializer import TOMLSerializer

warnings.simplefilter("ignore", UserWarning)
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)

mads_datasets.__version__

In [ ]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb", cache_dir="../assets/imdb/")
train = dataset["train"]
test = dataset["test"]

In [ ]:
from vectormesh import RegexVectorizer
from vectormesh.data.vectorizers import (
    build_imdb_review_pattern,
    harmonize_imdb_match
)


# Initialize & fit with training_texts
regexvectorizer = RegexVectorizer(
    pattern_builder=build_imdb_review_pattern,
    harmonizer=harmonize_imdb_match,
    min_doc_frequency=15,
    max_features=200,
    device="cpu",
    training_texts=train["text"],  # fit it on the full train set
)
regexvectorizer.get_metadata

In [ ]:
hidden_size = regexvectorizer.get_metadata["hidden_size"]

In [ ]:
regexvectorizer.print_stats()

In [ ]:
batch = train.shuffle()[:10]
batch

In [ ]:
regexvectorizer(batch['text'])

In [ ]:
from pathlib import Path    
from vectormesh import VectorCache

traincache = VectorCache.create(
    cache_dir=Path("tmp/artefacts"),
    vectorizer=regexvectorizer,
    dataset=train,
    dataset_tag="imdb_train",
)
testcache = VectorCache.create(
    cache_dir=Path("tmp/artefacts"),
    vectorizer=regexvectorizer,
    dataset=test,
    dataset_tag="imdb_test",
)

In [ ]:
traincache.metadata

In [ ]:
sample = traincache.select(range(100))
for batch in sample.iter(batch_size=5):
    feats = batch["regex_features"]
    print(feats)
    break

In [ ]:
from vectormesh.components import Serial, NeuralNet

pipeline = Serial([NeuralNet(hidden_size=hidden_size, out_size=2)]) 

In [ ]:
sample = traincache.select(range(100))
for batch in sample.iter(batch_size=5):
    feats = batch["regex_features"]
    predict = pipeline(feats)
    print(predict)
    break

In [ ]:
sample[0]

In [ ]:
from vectormesh.data import OneHot

onehot = OneHot(num_classes=2, label_col="label", target_col="onehot")
train_oh = traincache.map(onehot)
test_oh = testcache.map(onehot)

In [ ]:
from vectormesh.data import Collate
import torch

collate_fn = Collate(
    embedding_col="regex_features",
    target_col="onehot",
    padder=torch.stack
)

In [ ]:
from torch.utils.data import DataLoader

trainloader = DataLoader(train_oh, batch_size=32, shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(test_oh, batch_size=32, shuffle=False, collate_fn=collate_fn)

X, y = next(iter(trainloader))
X.shape, y.shape

In [ ]:
X.dtype, y.dtype

In [ ]:
from pathlib import Path

from mltrainer import ReportTypes, TrainerSettings

from vectormesh.components.metrics import F1Score, Accuracy

log_dir = Path("demo").absolute()

settings = TrainerSettings(
    epochs=2,
    metrics=[F1Score()],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(testloader),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)
settings

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
import torch.optim as optim
from mltrainer import Trainer

from vectormesh.data.vectorizers import detect_device

device = detect_device()
print(f"Using device: {device}")

trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=testloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

In [ ]:
trainer.loop()

In [ ]:
from vectormesh import Vectorizer
model_name = "ibm-granite/granite-embedding-small-english-r2"
vectorizer = Vectorizer(model_name=model_name, col_name="granite-small", max_length=512)


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

2026-06-07 16:58:31.175 | INFO     | vectormesh.data.vectorizers:initialize_model:126 - Using device: mps
2026-06-07 16:58:31.176 | INFO     | vectormesh.data.vectorizers:initialize_model:127 - using max_length: 512 (model max: 8192), stride: 51


PosixPath('/Users/rgrouls/code/courses/MADS-MachineLearning-course/artefacts/20260607154702_imdb_train_embed')

In [130]:
from vectormesh.types import VectorMeshError
artefacts = Path("../../artefacts").resolve()

trainpath = next(artefacts.glob("*imdb_train_embed"))
testpath = next(artefacts.glob("*imdb_test_embed"))

CREATED = False
try:
    traincache = VectorCache.load(trainpath)
    testcache = VectorCache.load(testpath)
    CREATED = True
except VectorMeshError as e:
    print(f"Error loading vector caches: {e}")


2026-06-07 17:05:23.920 | SUCCESS  | vectormesh.data.cache:load:140 - Cache loaded from /Users/rgrouls/code/courses/MADS-MachineLearning-course/artefacts/20260607154702_imdb_train_embed
2026-06-07 17:05:23.924 | SUCCESS  | vectormesh.data.cache:load:140 - Cache loaded from /Users/rgrouls/code/courses/MADS-MachineLearning-course/artefacts/20260607155309_imdb_test_embed


In [132]:
if not CREATED:
    traincache = VectorCache.create(
        cache_dir=Path("tmp/artefacts"),
        vectorizer=vectorizer,
        dataset=train_oh,
        dataset_tag="imdb_train",
    )
    testcache = VectorCache.create(
        cache_dir=Path("tmp/artefacts"),
        vectorizer=vectorizer,
        dataset=test_oh,
        dataset_tag="imdb_test",
    )
else:
    print("Vectorcaches loaded successfully, skipping recreation.")

Vectorcaches loaded successfully, skipping recreation.


In [ ]:
traincache.metadata

In [ ]:
hidden_size = traincache.metadata["embed"]["hidden_size"]
print(f"Hidden size: {hidden_size}")

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
column_name = "embed"
cnt = Counter()
for e in traincache.iter(batch_size=64):
    for tensor in e[column_name]:
        cnt[tensor.shape[0]] += 1

shapes = sorted(cnt.keys())
frequencies = [cnt[s] for s in shapes]

# Creating the plot
plt.figure(figsize=(10, 6))
plt.bar(shapes, frequencies, color="skyblue", edgecolor="black", alpha=0.8)

plt.xlabel("Tensor Shape (Dimension 0)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.title("Distribution of Tensor Shapes", fontsize=14)

In [ ]:
from vectormesh.components import FixedPadding

C=2
padder = FixedPadding(max_chunks=C)
i = 0
for batch in traincache.iter(batch_size=16):
    emb = padder(batch[column_name])
    print(emb.shape)
    i += 1
    if i == 2:
        break

In [ ]:
traincache

In [ ]:
from vectormesh.components import MaskedMeanAggregator

aggregator = MaskedMeanAggregator()
padder = FixedPadding(max_chunks=C)
for batch in traincache.iter(batch_size=16):
    emb = padder(batch[column_name])
    agg = aggregator(emb)
    print(agg.shape)
    break

In [ ]:
collate_fn = Collate(
    embedding_col="embed",
    target_col="onehot",
    padder=FixedPadding(max_chunks=C),
)

In [ ]:
pipeline = Serial([MaskedMeanAggregator(), NeuralNet(hidden_size=hidden_size, out_size=2)])

In [ ]:
trainloader = DataLoader(traincache, batch_size=32, shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(testcache, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
X, y = next(iter(trainloader))
X.shape, y.shape

In [ ]:
MaskedMeanAggregator()(X).shape

In [ ]:
pipeline(X).shape

In [ ]:
trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=testloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

In [ ]:
trainer.loop()